# QE Convergence Test (Need to update more)

All functions are in `convergence_test.py`. This notebook handles inputs and plotting.

In [ ]:
%matplotlib inline

In [ ]:

from ase.io import read
from ase.calculators.espresso import EspressoProfile
import matplotlib.pyplot as plt

from convergence_test import freeze_bottom_half, run_convergence


## Load structure

In [ ]:

atoms = read('/pscratch/sd/s/shikim/pyntatest/preprocessing/Ru.xyz')
atoms.center(vacuum=10)


## Pseudopotentials

In [ ]:

pseudopotentials = {
    'Ru': 'Ru.pbe-spn-kjpaw_psl.1.0.0.UPF',
    'O':  'O.pbe-n-rrkjus_psl.1.0.0.UPF',
    'H':  'H.pbe-rrkjus_psl.1.0.0.UPF'
}


## Base QE input

In [ ]:

base_input = {
    'calculation': 'scf',
    'tstress': True,
    'tprnfor': True,
    'disk_io': 'none',
    'occupations': 'smearing',
    'smearing': 'mp',
    'input_dft': 'beef-vdw',
    'degauss': 0.02,
    'nosym': True,
    'mixing_mode': 'local-TF',
    'conv_thr': 1e-5,
}


## QE execution profile

In [ ]:

profile = EspressoProfile(
    pseudo_dir='/global/cfs/cdirs/m4126/shikim/espresso/pseudo',
    command='pw.x -input espresso.pwi > espresso.pwo',
)


## Constraints

In [ ]:

atoms = freeze_bottom_half(atoms)


## Convergence parameters

In [ ]:

ecut_values = [30, 40, 50, 60, 70]
kmesh_values = [(3,3,1), (4,4,1), (5,5,1), (6,6,1)]


## Run convergence test

In [ ]:

results = run_convergence(
    atoms=atoms,
    base_input=base_input,
    pseudopotentials=pseudopotentials,
    profile=profile,
    ecut_values=ecut_values,
    kmesh_values=kmesh_values
)


## k-point convergence

In [ ]:

plt.figure()
for ecut in ecut_values:
    Es = [E for (ec, kp, E) in results if ec == ecut]
    ks = [k[0] for k in kmesh_values]
    plt.plot(ks, Es, marker='o', label=f'ecut = {ecut} Ry')

plt.xlabel('k-point grid (n × n × 1)')
plt.ylabel('Energy (eV)')
plt.title('k-point Convergence')
plt.legend()
plt.grid(True)
plt.show()


## ecut convergence

In [ ]:

fixed_k = kmesh_values[-1]
Es_fixed = [E for (ec, kp, E) in results if kp == fixed_k]

plt.figure()
plt.plot(ecut_values, Es_fixed, marker='o')
plt.xlabel('ecut (Ry)')
plt.ylabel('Energy (eV)')
plt.title(f'ecut Convergence at kpts={fixed_k}')
plt.grid(True)
plt.show()
